In [39]:
from pathlib import Path
from typing import Literal
from pydantic import BaseModel, Field

texts = []
for path in sorted(Path("data").glob("student_text_*.txt")):
    texts.append(path.read_text(encoding="utf-8"))

len(texts)

2

In [40]:
from pydantic_ai import Agent
from model import nvidia_model

model = nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b")

Level = Literal[0, 1, 2, 3]

class Student(BaseModel):
    proposed_grade: Literal["F", "E", "D", "C", "B", "A"] = Field(description="Grade on the Swedish scale, A highest, F lowest")
    innehall: Level = Field(description="Aspektnivå för Innehåll enligt bedömningsmatrisen")
    disposition: Level = Field(description="Aspektnivå för Disposition enligt bedömningsmatrisen")
    sprak: Level = Field(description="Aspektnivå för Språk enligt bedömningsmatrisen")
    motivation: str = Field(description="Motivering per aspekt, 1-2 meningar var")
    improvements: str = Field(description="Vad eleven kan förbättra, 1-2 meningar")

grading_agent = Agent(
    model=model,
    instructions=("Du är en erfaren svensklärare på gymnasiet. "
                "Du bedömer elevtexter och ger ett betygförslag. Svara på svenska. "
    ),
)

async def grade_text(text: str) -> Student:
    result = await grading_agent.run(text, output_type=Student)
    return result.output

In [41]:
criteria_agent = Agent(
    model=model,
    instructions=(
        "Du är en erfaren svensklärare på gymnasiet"
        "Du bedömer elevtexter och ger betygsförslag. Svara på svenska"
        "Bedöm texten enbart utifrån följande betygskriterier:\n\n"
        + criteria
    ),
)

async def grade_with_criteria(text: str) -> Student:
    result = await criteria_agent.run(text, output_type=Student)
    return result.output

In [42]:
for i, text in enumerate(texts, start=1):
    grade = await grade_with_criteria(text)

    folder = Path("grades_med_kriterier") / f"student_{i}"
    folder.mkdir(parents=True, exist_ok=True)

    (folder / "proposed_grade.txt").write_text(grade.proposed_grade, encoding="utf-8")
    (folder / "motivation.txt").write_text(grade.motivation, encoding="utf-8")
    (folder / "improvements.txt").write_text(grade.improvements, encoding="utf-8")

In [43]:
all_text = Path("data/svenska1_criterias.txt").read_text(encoding="utf-8")

start = all_text.find("Betygskriterier")
criteria = all_text[start:]

print(criteria)


Betygskriterier
Betyget E
Eleven kan, i förberedda samtal och diskussioner, muntligt förmedla egna tankar och åsikter samt genomföra muntlig framställning inför en grupp. Detta gör eleven med viss säkerhet. Den muntliga framställningen är sammanhängande och begriplig. Eleven har viss åhörarkontakt. Språket är till viss del anpassat till syfte, mottagare och kommunikations­situation. Vidare kan eleven med viss säkerhet använda presentationstekniska hjälpmedel.

Eleven kan skriva argumenterande text och andra typer av texter, som är sammanhängande och begripliga samt till viss del anpassade till syfte, mottagare och kommunikations­situation. Eleven kan i huvudsak följa skriftspråkets normer för språkriktighet.

Eleven kan läsa, reflektera över och göra enkla sammanfattningar av texter samt skriva egna texter som anknyter till det lästa. I sitt arbete värderar och granskar eleven med viss säkerhet källor kritiskt samt tillämpar grundläggande regler för citat- och referatteknik.

Eleven ka

In [44]:
for i, text in enumerate(texts, start=1):
    without = await grade_text(text)
    with_criteria = await grade_with_criteria(text)

    print(f"Elev {i}")
    print(" utan kriterier:", without.proposed_grade)
    print(" med kriterier:", with_criteria.proposed_grade)


Elev 1
 utan kriterier: E
 med kriterier: E
Elev 2
 utan kriterier: E
 med kriterier: E


In [45]:
task = Path("data/uppgift.txt").read_text(encoding="utf-8")
matrix = Path("data/bedomningsmatris.txt").read_text(encoding="utf-8")

def make_agent(example_text: str, example_levels: str) -> Agent:
    return Agent(
        model=model,
        instructions=(
            "Du är en erfaren svensklärare och bedömer nationella prov. Svara på svenska.\n\n"
            "Eleverna fick denna uppgift:\n" + task +
            "\n\nBedöm enligt denna matris:\n" + matrix +
            "\n\nExempel på en bedömd elevtext:\n" + example_text +
            "\n\nLärarens bedömning: " + example_levels
        ),
    )

agent_for_elev_2 = make_agent(texts[0], "Innehåll 1, Disposition 1, Språk 1")
agent_for_elev_1 = make_agent(texts[1], "Innehåll 1, Disposition 1, Språk 2")

result_1 = await agent_for_elev_1.run(texts[0], output_type=Student)
result_2 = await agent_for_elev_2.run(texts[1], output_type=Student)

for name, result, facit in [("Elev 1", result_1, "1 / 1 / 1"), ("Elev 2", result_2, "1 / 1 / 2")]:
    s = result.output
    print(name, "LLM:", s.innehall, "/", s.disposition, "/", s.sprak, "  Lärare:", facit)

Elev 1 LLM: 1 / 1 / 1   Lärare: 1 / 1 / 1
Elev 2 LLM: 2 / 2 / 1   Lärare: 1 / 1 / 2
